In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("UrbanFlow - Data Profiling") \
    .getOrCreate()

RAW_PATH = "gs://trip-data-project-e6d9c504-e176-49d1-acd/urbanflow/raw/hvfhv/"

df = spark.read.parquet(RAW_PATH)

print("=== URBANFLOW - DATA PROFILING ===")
print(f"Columnas: {len(df.columns)}")

df.printSchema()

total_records = df.count()

print(f"Registros RAW: {total_records:,}")

In [ ]:
# ============================================================
# 1. RANGO TEMPORAL
# ============================================================

date_profile = df.select(
    F.min("pickup_datetime").alias("min_pickup_datetime"),
    F.max("pickup_datetime").alias("max_pickup_datetime"),
    F.min("dropoff_datetime").alias("min_dropoff_datetime"),
    F.max("dropoff_datetime").alias("max_dropoff_datetime")
)

print("=== RANGO TEMPORAL ===")
date_profile.show(truncate=False)


# ============================================================
# 2. NULOS EN CAMPOS CRÍTICOS
# ============================================================

critical_columns = [
    "request_datetime",
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_miles",
    "trip_time",
    "base_passenger_fare",
    "driver_pay"
]

null_exprs = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in critical_columns
]

print("=== NULOS EN CAMPOS CRÍTICOS ===")
df.select(null_exprs).show(truncate=False)


# ============================================================
# 3. REGLAS DE CALIDAD
# ============================================================

quality_profile = df.agg(

    F.sum(
        F.when(F.col("trip_miles") <= 0, 1).otherwise(0)
    ).alias("invalid_trip_miles"),

    F.sum(
        F.when(F.col("trip_time") <= 0, 1).otherwise(0)
    ).alias("invalid_trip_time"),

    F.sum(
        F.when(F.col("base_passenger_fare") < 0, 1).otherwise(0)
    ).alias("negative_fare"),

    F.sum(
        F.when(F.col("driver_pay") < 0, 1).otherwise(0)
    ).alias("negative_driver_pay"),

    F.sum(
        F.when(
            F.col("pickup_datetime") > F.col("dropoff_datetime"),
            1
        ).otherwise(0)
    ).alias("pickup_after_dropoff"),

    F.sum(
        F.when(
            F.col("request_datetime") > F.col("pickup_datetime"),
            1
        ).otherwise(0)
    ).alias("request_after_pickup")
)

print("=== CALIDAD DE DATOS ===")
quality_profile.show(truncate=False)


# ============================================================
# 4. ESTADÍSTICAS BÁSICAS
# ============================================================

print("=== ESTADÍSTICAS DESCRIPTIVAS ===")

df.select(
    "trip_miles",
    "trip_time",
    "base_passenger_fare",
    "driver_pay"
).describe().show(truncate=False)


# ============================================================
# 5. DISTRIBUCIÓN DE REGISTROS POR AÑO Y MES
# ============================================================

monthly_records = (
    df
    .withColumn("year", F.year("pickup_datetime"))
    .withColumn("month", F.month("pickup_datetime"))
    .groupBy("year", "month")
    .agg(F.count("*").alias("trip_count"))
    .orderBy("year", "month")
)

print("=== REGISTROS POR AÑO Y MES ===")
monthly_records.show(50, truncate=False)

In [ ]:
# ============================================================
# RESUMEN EJECUTIVO DEL PROFILING
# ============================================================

summary = df.agg(

    F.count("*").alias("raw_records"),

    F.min("pickup_datetime").alias("min_pickup_datetime"),
    F.max("pickup_datetime").alias("max_pickup_datetime"),

    F.sum(
        F.when(F.col("trip_miles") <= 0, 1).otherwise(0)
    ).alias("invalid_trip_miles"),

    F.sum(
        F.when(F.col("trip_time") <= 0, 1).otherwise(0)
    ).alias("invalid_trip_time"),

    F.sum(
        F.when(F.col("base_passenger_fare") < 0, 1).otherwise(0)
    ).alias("negative_fare"),

    F.sum(
        F.when(F.col("driver_pay") < 0, 1).otherwise(0)
    ).alias("negative_driver_pay"),

    F.sum(
        F.when(
            F.col("pickup_datetime") > F.col("dropoff_datetime"),
            1
        ).otherwise(0)
    ).alias("pickup_after_dropoff"),

    F.sum(
        F.when(
            F.col("request_datetime") > F.col("pickup_datetime"),
            1
        ).otherwise(0)
    ).alias("request_after_pickup")
)

summary.show(truncate=False)

print("=== RESULTADOS ESPERADOS DEL PROYECTO ===")
print("Registros RAW esperados: 710,576,363")
print("Columnas RAW esperadas: 24")
print("Periodo esperado: septiembre 2023 - julio 2026")